# **Day 75: Databases with SQLite**

A **database** stores data in an organized way, so that we can **search**, **add**, **change** and **delete** it easily, even when there is a lot of data.

**SQLite** is a complete database stored in **one single file**. Python has it **built in** (the `sqlite3` module), so nothing needs to be installed.

We talk to a database using **SQL** (Structured Query Language). SQL commands look like simple English:
- `CREATE TABLE`: make a new table
- `INSERT`: add rows
- `SELECT`: read rows
- `UPDATE`: change rows
- `DELETE`: remove rows

**Did you know?** The popular GIS format **GeoPackage (.gpkg)**, used a lot in QGIS, is actually an SQLite database inside!

In [1]:
import sqlite3

conn = sqlite3.connect("bengal.db")      # Creates the file if it does not exist
cur = conn.cursor()
print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.50.4


## **Creating Tables**
A table is like an Excel sheet: it has **columns** (with fixed types) and **rows**.

In [2]:
cur.executescript("""
DROP TABLE IF EXISTS districts;
DROP TABLE IF EXISTS scenes;

CREATE TABLE districts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE,
    hq TEXT,
    region TEXT CHECK (region IN ('north', 'south', 'west')),
    population INTEGER,
    lat REAL,
    lon REAL
);

CREATE TABLE scenes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    scene_id TEXT NOT NULL,
    district_id INTEGER REFERENCES districts(id),
    satellite TEXT,
    date TEXT,
    cloud REAL
);
""")
conn.commit()

- **PRIMARY KEY:** A unique ID for every row.
- **NOT NULL:** This value cannot be empty.
- **UNIQUE:** No two rows can have the same value.
- **CHECK:** Only allowed values can be saved.
- **REFERENCES:** Links a row to a row in another table.

## **Adding Data (INSERT)**
**Always use `?` placeholders** for values. Never build SQL text with f-strings from user input (the danger is shown at the end).

In [3]:
districts = [
    ("Kolkata", "Kolkata", "south", 4496694, 22.5726, 88.3639),
    ("Darjeeling", "Darjeeling", "north", 1846823, 27.0410, 88.2663),
    ("Paschim Medinipur", "Midnapore", "south", 5913457, 22.4257, 87.3199),
    ("Purba Medinipur", "Tamluk", "south", 5095875, 22.3000, 87.9200),
    ("Jalpaiguri", "Jalpaiguri", "north", 3872846, 26.5200, 88.7200),
    ("Bankura", "Bankura", "west", 3596674, 23.2300, 87.0700),
    ("Purulia", "Purulia", "west", 2930115, 23.3300, 86.3600),
]
cur.executemany("INSERT INTO districts (name, hq, region, population, lat, lon) VALUES (?, ?, ?, ?, ?, ?)", districts)

scenes = [
    ("S2B_45QXF_20240120", "Kolkata", "Sentinel-2", "2024-01-20", 0.4),
    ("LC08_138044_20231110", "Kolkata", "Landsat 8", "2023-11-10", 2.3),
    ("S2B_45RXL_20241205", "Darjeeling", "Sentinel-2", "2024-12-05", 5.5),
    ("S2A_45RXL_20240810", "Darjeeling", "Sentinel-2", "2024-08-10", 64.0),
    ("LC08_139045_20240214", "Paschim Medinipur", "Landsat 8", "2024-02-14", 7.8),
    ("S2A_45QWE_20240301", "Purba Medinipur", "Sentinel-2", "2024-03-01", 12.0),
    ("LC09_139044_20240110", "Bankura", "Landsat 9", "2024-01-10", 1.1),
    ("LC09_140044_20240117", "Purulia", "Landsat 9", "2024-01-17", 0.0),
]
for sid, district, sat, date, cloud in scenes:
    cur.execute("""INSERT INTO scenes (scene_id, district_id, satellite, date, cloud)
                   VALUES (?, (SELECT id FROM districts WHERE name = ?), ?, ?, ?)""",
                (sid, district, sat, date, cloud))
conn.commit()
print(cur.execute("SELECT COUNT(*) FROM scenes").fetchone()[0], "scenes saved")

8 scenes saved


## **Reading Data (SELECT)**

In [4]:
for row in cur.execute("SELECT name, hq, population FROM districts ORDER BY population DESC LIMIT 3"):
    print(row)

('Paschim Medinipur', 'Midnapore', 5913457)
('Purba Medinipur', 'Tamluk', 5095875)
('Kolkata', 'Kolkata', 4496694)


In [5]:
cur.execute("SELECT name FROM districts WHERE region = ?", ("north",))
print(cur.fetchall())

[('Darjeeling',), ('Jalpaiguri',)]


## **Rows as Dictionaries**
With `row_factory = sqlite3.Row`, we can use column names instead of positions.

In [6]:
conn.row_factory = sqlite3.Row
cur = conn.cursor()
r = cur.execute("SELECT * FROM districts WHERE name = ?", ("Bankura",)).fetchone()
print(dict(r))
print(r["hq"], r["lat"], r["lon"])

{'id': 6, 'name': 'Bankura', 'hq': 'Bankura', 'region': 'west', 'population': 3596674, 'lat': 23.23, 'lon': 87.07}
Bankura 23.23 87.07


## **Joining Two Tables (JOIN)**
`JOIN` connects rows of two tables using the linked ID.

In [7]:
query = """
SELECT s.scene_id, s.satellite, s.date, s.cloud, d.name AS district
FROM scenes s
JOIN districts d ON d.id = s.district_id
WHERE s.cloud < ?
ORDER BY s.cloud
"""
print("Clear scenes (cloud below 5%):")
for row in cur.execute(query, (5,)):
    print(f"  {row['scene_id']:<22}{row['satellite']:<11}{row['date']}  {row['cloud']:>4}%  {row['district']}")

Clear scenes (cloud below 5%):
  LC09_140044_20240117  Landsat 9  2024-01-17   0.0%  Purulia
  S2B_45QXF_20240120    Sentinel-2 2024-01-20   0.4%  Kolkata
  LC09_139044_20240110  Landsat 9  2024-01-10   1.1%  Bankura
  LC08_138044_20231110  Landsat 8  2023-11-10   2.3%  Kolkata


## **Grouping (GROUP BY)**

In [8]:
query = """
SELECT d.region, COUNT(s.id) AS scenes, ROUND(AVG(s.cloud), 1) AS avg_cloud
FROM scenes s JOIN districts d ON d.id = s.district_id
GROUP BY d.region
ORDER BY scenes DESC
"""
for row in cur.execute(query):
    print(dict(row))

{'region': 'south', 'scenes': 4, 'avg_cloud': 5.6}
{'region': 'west', 'scenes': 2, 'avg_cloud': 0.6}
{'region': 'north', 'scenes': 2, 'avg_cloud': 34.8}


## **Example from GIS: A Simple Spatial Search**
Find all district headquarters within a **bounding box** around South-West Bengal. (Real spatial databases like PostGIS and SpatiaLite have special functions for this, but a simple box search already works with plain SQL.)

In [9]:
box = {"min_lat": 22.0, "max_lat": 23.5, "min_lon": 86.0, "max_lon": 87.5}

rows = cur.execute("""SELECT name, hq, lat, lon FROM districts
                      WHERE lat BETWEEN :min_lat AND :max_lat
                        AND lon BETWEEN :min_lon AND :max_lon""", box).fetchall()
for r in rows:
    print(f"{r['hq']:<10} ({r['lat']}, {r['lon']})  in {r['name']}")

Midnapore  (22.4257, 87.3199)  in Paschim Medinipur
Bankura    (23.23, 87.07)  in Bankura
Purulia    (23.33, 86.36)  in Purulia


- `:min_lat` is a **named placeholder**. Its value comes from the dictionary `box`.

## **Updating and Deleting**

In [10]:
cur.execute("UPDATE scenes SET cloud = ? WHERE scene_id = ?", (3.0, "S2A_45QWE_20240301"))
print("Rows updated:", cur.rowcount)

cur.execute("DELETE FROM scenes WHERE cloud > ?", (50,))
print("Cloudy scenes deleted:", cur.rowcount)
conn.commit()

Rows updated: 1
Cloudy scenes deleted: 1


## **Transactions: All or Nothing**
Using `with conn:` saves all changes if everything works, and **cancels all of them** if any step fails.

In [11]:
try:
    with conn:
        conn.execute("INSERT INTO districts (name, hq, region) VALUES (?, ?, ?)", ("Nadia", "Krishnanagar", "south"))
        conn.execute("INSERT INTO districts (name, hq, region) VALUES (?, ?, ?)", ("Malda", "English Bazar", "east"))
except sqlite3.IntegrityError as e:
    print("Cancelled:", e)

print("Nadia saved?", conn.execute("SELECT COUNT(*) FROM districts WHERE name = 'Nadia'").fetchone()[0] == 1)

Cancelled: CHECK constraint failed: region IN ('north', 'south', 'west')
Nadia saved? False


The second insert broke the CHECK rule ('east' is not allowed), so the first insert (Nadia) was also cancelled.

## **Why ? Placeholders Matter: SQL Injection**
If we build SQL text by joining strings, a trick input can change the meaning of the query. This is called **SQL injection**, and it is one of the most common security problems.

In [12]:
trick = "x' OR '1'='1"

unsafe = f"SELECT name FROM districts WHERE name = '{trick}'"
print("Unsafe query returns:", len(conn.execute(unsafe).fetchall()), "rows (all of them!)")

safe = conn.execute("SELECT name FROM districts WHERE name = ?", (trick,)).fetchall()
print("Safe query returns  :", len(safe), "rows")

Unsafe query returns: 7 rows (all of them!)
Safe query returns  : 0 rows


In [13]:
conn.close()

In [14]:
# Quick question
# Which SQL command do we use to read data from a table?